# Superstore Sales Analysis
**Goal~ SQL Foundations**

This notebook analyses a retail superstore dataset using SQL queries via `pandasql`.

Business Insights being addressed are:
- Which regions generate the most sales?
- Which product categories are most profitable?
- Which products generate the highest revenue?
- Do discounts actually hurt profit?

**Dataset:** Sample Superstore (Kaggle)  
**Tools:** Python, pandas, pandasql

In [ ]:
import pandas as pd
from pandasql import sqldf

In [ ]:
df = pd.read_csv("Sample - Superstore.csv", encoding='latin-1')


In [ ]:
print(df.columns.tolist())
df.head()


['Row ID', 
'Order ID', 
'Order Date', 
'Ship Date', 
'Ship Mode', 
'Customer ID',
'Customer Name',
'Segment', 
'Country', 
'City', 
'State', 
'Postal Code', '
Region', 
'Product ID', 
'Category', 
'Sub-Category',
'Product Name',
'Sales',
'Quantity',
'Discount',
'Profit']

In [ ]:
q1= "SELECT * FROM df limit 10;"
sqldf(q1, locals())

**What did I do**
- i tried previewing the data of first day rows.

---
## Query 2 — Sales by Region

**Business Insight:** Which geographic regions are driving the most revenue?

**Importance:** Regional performance tells management where to invest more resources 
and where to investigate underperformance. A region with high sales but low profit 
may have operational or discount problems.

**SQL concepts used:** `GROUP BY`, `SUM()`, `ROUND()`, `ORDER BY DESC`

In [7]:
q2 = """
SELECT Region, 
       Round(SUM(SALES), 2) AS total_sales
FROM df
GROUP BY Region
ORDER BY total_sales DESC;
"""
sqldf(q2, locals())


,Region,total_sales
0,West,725457.82
1,East,678781.24
2,Central,501239.89
3,South,391721.91


___
### Query 2 Insight
The *West region* generates the highest total sales of 725,000 followed by **East** region total sales 678781.24

The **South** region lags behind at 391721.91 ( the management should investigate whether this reflects lower demand, fewer sales reps or a pricing issue in  that territory.

Management should investigate South region performance.



---
## Query 3 — Category Profitability

**Business Insight:** Which product categories generate the most profit, and what is the profit margin for each?

**Importance:** A category can have high sales but low profit if costs are high 
or discounts are excessive. Profit margin % reveals the true health of each category 
and guides pricing and procurement decisions.

**SQL concepts used:** `SUM()`, `ROUND()`, calculated column (profit margin %), `ORDER BY`

In [8]:
q3 = """
SELECT Category,
       ROUND(SUM(Profit), 2) AS total_profit, 
       ROUND(SUM(Sales), 2) AS total_sales,
       ROUND((SUM (Profit)/SUM(Sales))*100 ,2) AS profit_margin_pct

FROM df
Group BY Category
ORDER BY total_profit DESC;
"""
sqldf(q3,locals())

,Category,total_profit,total_sales,profit_margin_pct
0,Technology,145454.95,836154.03,17.40
1,Office Supplies,122490.80,719047.03,17.04
2,Furniture,18451.27,741999.80,2.49


### Insight — Query 3

**Technology** has the highest total profit of 145454.95 but **Office Supplies** may show a stronger 
profit margin percentage of 17.04, meaning it is more efficient per dollar sold.  
**Furniture** often shows surprisingly low margins despite high sales — this means that it is  a 
common red flag in retail and worth flagging to management.

---
## Query 4 — Top 10 Products by Revenue

**Business Question:** Which individual products generate the highest total revenue, and how frequently are they ordered?

**Importance:** High-revenue products should never go out of stock. 
Cross-referencing revenue with order frequency reveals whether a product is expensive 
but rarely bought, or cheap but ordered constantly — two very different business stories.

**SQL concepts used:** `GROUP BY`, `SUM()`, `COUNT(*)`, `LIMIT`, backtick quoting for column names with spaces

In [11]:
q4 = """
SELECT `Product Name`,
       ROUND(SUM(SALES), 2) AS total_revenue,
       COUNT (*) AS times_Ordered
FROM df
GROUP BY `Product Name`
ORDER BY total_revenue DESC
LIMIT 10;
"""
sqldf(q4, locals())

,Product Name,total_revenue,times_Ordered
0,Canon imageCLASS 2200 Advanced Copier,61599.82,5
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,10
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48,1
3,HON 5400 Series Task Chairs for Big and Tall,21870.58,8
4,GBC DocuBind TL300 Electric Binding System,19823.48,11
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.50,9
6,Hewlett Packard LaserJet 3310 Copier,18839.69,8
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.90,3
8,GBC DocuBind P400 Electric Binding System,17965.07,6
9,High Speed Automatic Electric Letter Opener,17030.31,3


### Insight — Query 4

**Replace this with your actual findings after running the query.**

**Example format:**  
The top revenue products are predominantly from the **Technology** category, 
particularly copiers and phones. Notably, some high-revenue products have a 
low number of orders which means they are expensive single purchases rather 
than repeat buys.
Inventory strategy for these should prioritise reliability over volume.

---
## Query 5 — Discount Impact on Profit

**Business Insight:** Are discounts helping sales or destroying profit? ( this is a very important fundamental question in business)

**Importance:** Many retailers assume discounts drive growth, but aggressive discounting 
often leads to negative profit margins. This query segments orders by discount level 
and reveals the average profit at each tier — a critical insight for pricing strategy.

**SQL concepts used:** `CASE WHEN` (conditional logic), `GROUP BY`, `AVG()`, `COUNT(*)`

In [10]:
q5 = """
SELECT 
    CASE 
        WHEN Discount = 0 THEN 'No Discount'
        WHEN Discount <= 0.2 THEN 'Low Discount'
        WHEN Discount <= 0.4 THEN 'Medium Discount'
        ELSE 'High Discount'
    END AS discount_tier,
    COUNT(*) AS order_count,
    ROUND(AVG(Profit), 2) AS avg_profit
FROM df
GROUP BY discount_tier
ORDER BY avg_profit DESC;
"""
sqldf(q5, locals())

,discount_tier,order_count,avg_profit
0,No Discount,4798,66.90
1,Low Discount,3803,26.50
2,Medium Discount,460,-77.86
3,High Discount,933,-106.71


### Insight — Query 5

As discount level increases, average profit drops 
and **High Discount** orders likely show a **negative average profit**, meaning the 
business loses money on every heavily discounted saleNo discount earns an avg profit of 66.90 
Low discount drastically dipped the profit at 26.50
medium and high discounts resulted in avgerage profit in negative which means loss occured when given medium to high profits.

Management should Cap discounts at 20% and review the pricing strategy for 
any product currently offered at 40%+ discount.
